In [3]:
import numpy as np
import pandas as pd
import scanpy as sc

adata = sc.read_h5ad("/data1/xiaoxinyu/benchmark/DLPFC/process-data/train-data/151509/st_data.h5ad")

# 若是稀疏矩阵
X_dense = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X


In [14]:
print(X_dense.shape)

(4789, 33538)


In [ ]:
import gseapy as gp
from gseapy.plot import barplot
from tqdm import tqdm

# 使用 MSigDB 的 hallmark 基因集（也可以下载 GO_Biological_Process）
gene_sets = gp.get_library_name(organism='Human')

print("可用的基因集:", gene_sets)
# 推荐使用: 'GO_Biological_Process_2025', 'KEGG_2021_Human'


可用的基因集: ['ARCHS4_Cell-lines', 'ARCHS4_IDG_Coexp', 'ARCHS4_Kinases_Coexp', 'ARCHS4_TFs_Coexp', 'ARCHS4_Tissues', 'Achilles_fitness_decrease', 'Achilles_fitness_increase', 'Aging_Perturbations_from_GEO_down', 'Aging_Perturbations_from_GEO_up', 'Allen_Brain_Atlas_10x_scRNA_2021', 'Allen_Brain_Atlas_down', 'Allen_Brain_Atlas_up', 'Azimuth_2023', 'Azimuth_Cell_Types_2021', 'BioCarta_2013', 'BioCarta_2015', 'BioCarta_2016', 'BioPlanet_2019', 'BioPlex_2017', 'CCLE_Proteomics_2020', 'COMPARTMENTS_Curated_2025', 'COMPARTMENTS_Experimental_2025', 'CORUM', 'COVID-19_Related_Gene_Sets', 'COVID-19_Related_Gene_Sets_2021', 'Cancer_Cell_Line_Encyclopedia', 'CellMarker_2024', 'CellMarker_Augmented_2021', 'ChEA_2013', 'ChEA_2015', 'ChEA_2016', 'ChEA_2022', 'Chromosome_Location', 'Chromosome_Location_hg19', 'ClinVar_2019', 'DGIdb_Drug_Targets_2024', 'DSigDB', 'Data_Acquisition_Method_Most_Popular_Genes', 'DepMap_CRISPR_GeneDependency_CellLines_2023', 'DepMap_WG_CRISPR_Screens_Broad_CellLines_2019', 'Dep

### ssGSEA 分析

In [7]:
# 选择一个
geneset_name ='KEGG_2021_Human'

# 准备数据（每列为一个 spot，每行为一个基因）
# AnnData 默认 X 是 sample × gene，但 GSEApy 要求 gene × sample
expr_df = pd.DataFrame(X_dense.T, index=adata.var_names, columns=adata.obs_names)

# ssGSEA 分析
ssgsea_res = gp.ssgsea(
    data=expr_df,
    gene_sets=geneset_name, 
    sample_norm_method='rank',
    outdir=None,
    verbose=True
)

# 输出：ssgsea_res.res2d 是 (pathway × spot) 的打分表
scores = ssgsea_res.res2d.T  # shape: (spot, pathway)


2025-08-01 12:07:00,223 [INFO] Parsing data files for ssGSEA...........................
2025-08-01 12:07:13,915 [INFO] Downloading and generating Enrichr library gene sets......
2025-08-01 12:07:16,237 [INFO] 0014 gene_sets have been filtered out when max_size=500 and min_size=15
2025-08-01 12:07:16,238 [INFO] 0306 gene_sets used for further statistical testing.....
2025-08-01 12:07:16,239 [INFO] Start to run ssGSEA...Might take a while................


In [17]:
print(ssgsea_res.res2d.head())

                 Name                       Term            ES       NES
0  AACGCTGTTGCTGAAA-1                   Ribosome  12434.149377  0.606936
1  TATAGTTAGGTGTACT-1                   Ribosome  12385.476799   0.60456
2  GAATGAAGGTCTTCAG-1                   Ribosome  12376.156511  0.604105
3  TTATCCGGGATCTATA-1                   Ribosome  12368.762098  0.603744
4  TGAGTGTAACAACGGG-1  Oxidative phosphorylation  12348.040703  0.602733


In [20]:
scores_df = ssgsea_res.res2d.pivot(index='Term', columns='Name', values='ES').T
print(scores_df.shape)

(4789, 306)


In [ ]:
print(scores_df.head())

Term               ABC transporters  \
Name                                  
AAACAAGTATCTCCCA-1       -217.80952   
AAACAATCTACTAGCA-1      -269.500321   
AAACACCAATAACTGC-1       -87.444944   
AAACAGAGCGACTCCT-1       208.365973   
AAACAGCTTTCAGAAG-1      -946.878984   

Term               AGE-RAGE signaling pathway in diabetic complications  \
Name                                                                      
AAACAAGTATCTCCCA-1                                        1130.814801     
AAACAATCTACTAGCA-1                                          460.35672     
AAACACCAATAACTGC-1                                         741.187593     
AAACAGAGCGACTCCT-1                                        1334.613936     
AAACAGCTTTCAGAAG-1                                        3383.492181     

Term               AMPK signaling pathway Acute myeloid leukemia  \
Name                                                               
AAACAAGTATCTCCCA-1             1673.06806            2262.524639

### Top-K 通路 → 自然语言

In [23]:
def generate_text_descriptions(scores_df, k=5):
    """
    输入:
        scores_df: pd.DataFrame, shape = (n_samples, n_pathways)
        k: int, 选取 top-k 的通路

    输出:
        descriptions: List[str], 每个 sample 对应一条自然语言描述
    """
    descriptions = []

    for idx, row in scores_df.iterrows():
        top_pathways = row.sort_values(ascending=False).head(k).index.tolist()

        desc = [f"The '{path}' pathway is highly enriched." for path in top_pathways]
        full_sentence = " ".join(desc)

        descriptions.append(full_sentence)

    return descriptions


In [24]:
descriptions = generate_text_descriptions(scores_df, k=5)

# 示例输出前3个描述：
for i in range(3):
    print(f"[Sample {i}] {descriptions[i]}")


[Sample 0] The 'Ribosome' pathway is highly enriched. The 'Oxidative phosphorylation' pathway is highly enriched. The 'Phenylalanine metabolism' pathway is highly enriched. The 'Endocrine and other factor-regulated calcium reabsorption' pathway is highly enriched. The 'Herpes simplex virus 1 infection' pathway is highly enriched.
[Sample 1] The 'Ribosome' pathway is highly enriched. The 'Oxidative phosphorylation' pathway is highly enriched. The 'Endocrine and other factor-regulated calcium reabsorption' pathway is highly enriched. The 'Synaptic vesicle cycle' pathway is highly enriched. The 'Parkinson disease' pathway is highly enriched.
[Sample 2] The 'Ribosome' pathway is highly enriched. The 'Oxidative phosphorylation' pathway is highly enriched. The 'Endocrine and other factor-regulated calcium reabsorption' pathway is highly enriched. The 'Parkinson disease' pathway is highly enriched. The 'Synaptic vesicle cycle' pathway is highly enriched.
